In [17]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!apt-get install -y -qq openmpi-bin libopenmpi-dev
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-5d365dc5-35ff-82b3-ce91-bf085d8f2f7b, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [18]:
GITHUB_USERNAME = "AliOrynbay"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: AliOrynbay


In [19]:
%%writefile distributed_cuda_solver.py
import os
import sys
import re
import json
import math
import time
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15
KAPPA_MAX_THEORY = 0.24
CFL_SAFETY = 0.8


def get_gpu_uuid() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"], encoding="utf-8")
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf8")).hexdigest()
    return int(digest[:16], 16)


# ==============================================================================
# TASK 1
# ==============================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {size}.")

    cuda.select_device(0)
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    cart_comm = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    coords = cart_comm.Get_coords(rank)
    nbr_up, nbr_down = cart_comm.Shift(0, 1)      # (source=-1, dest=+1) along axis 0
    nbr_left, nbr_right = cart_comm.Shift(1, 1)   # along axis 1

    return {
        "comm": comm, "cart_comm": cart_comm, "rank": rank, "size": size,
        "coords": tuple(coords), "gpu_uuid": gpu_uuid, "seed": instance_seed,
        "up": nbr_up, "down": nbr_down, "left": nbr_left, "right": nbr_right,
        "ny": GLOBAL_NY // 2, "nx": GLOBAL_NX // 2,
    }


# ==============================================================================
# TASK 2
# ==============================================================================
def initialize_subdomains(ctx):
    ny, nx = ctx["ny"], ctx["nx"]
    row_rank, col_rank = ctx["coords"]
    dx = 1.0 / (GLOBAL_NX + 1)   # 1/513
    dy = 1.0 / (GLOBAL_NY + 1)

    # Same seed on every rank -> identical field parameters, no communication needed.
    rng = np.random.default_rng(ctx["seed"] % (2**63))
    ph1, ph2 = rng.uniform(0.0, 1.0, 2)
    centers = rng.uniform(0.2, 0.8, size=(3, 2))
    amps = rng.uniform(0.5, 1.0, size=3)
    src_c = rng.uniform(0.3, 0.7, size=2)

    gi = row_rank * ny + np.arange(ny + 2)   # global row index 0..513
    gj = col_rank * nx + np.arange(nx + 2)
    Y = (gi * dy)[:, None]
    X = (gj * dx)[None, :]

    kappa = 0.16 + 0.08 * np.sin(2 * np.pi * (3.0 * X + ph1)) * np.cos(2 * np.pi * (2.0 * Y + ph2))
    u = np.zeros((ny + 2, nx + 2))
    for (cx, cy), a in zip(centers, amps):
        u += a * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2 * 0.06 ** 2))
    S = 100.0 * np.exp(-((X - src_c[0]) ** 2 + (Y - src_c[1]) ** 2) / (2 * 0.05 ** 2))

    if ctx["up"] == MPI.PROC_NULL:
        u[0, :] = 0.0
    if ctx["down"] == MPI.PROC_NULL:
        u[ny + 1, :] = 0.0
    if ctx["left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if ctx["right"] == MPI.PROC_NULL:
        u[:, nx + 1] = 0.0

    return (np.ascontiguousarray(u), np.ascontiguousarray(kappa),
            np.ascontiguousarray(S), dx, dy)


# ==============================================================================
# TASK 3: pack / unpack kernels and async exchange
# ==============================================================================
@cuda.jit
def pack_halos_kernel(u, ny, nx, s_up, s_down, s_left, s_right):
    idx = cuda.grid(1)
    if idx < nx:
        s_up[idx] = u[1, idx + 1]
        s_down[idx] = u[ny, idx + 1]
    if idx < ny:
        s_left[idx] = u[idx + 1, 1]
        s_right[idx] = u[idx + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, ny, nx, r_up, r_down, r_left, r_right,
                        has_up, has_down, has_left, has_right):
    idx = cuda.grid(1)
    if idx < nx:
        if has_up:
            u[0, idx + 1] = r_up[idx]
        if has_down:
            u[ny + 1, idx + 1] = r_down[idx]
    if idx < ny:
        if has_left:
            u[idx + 1, 0] = r_left[idx]
        if has_right:
            u[idx + 1, nx + 1] = r_right[idx]


class HaloBuffers:
    def __init__(self, ny, nx):
        sizes = {"up": nx, "down": nx, "left": ny, "right": ny}
        self.d_send = {k: cuda.device_array(n, dtype=np.float64) for k, n in sizes.items()}
        self.d_recv = {k: cuda.device_array(n, dtype=np.float64) for k, n in sizes.items()}
        self.h_send = {k: cuda.pinned_array(n, dtype=np.float64) for k, n in sizes.items()}
        self.h_recv = {k: cuda.pinned_array(n, dtype=np.float64) for k, n in sizes.items()}
        for k in sizes:
            self.h_recv[k][:] = 0.0


# tag t: 0 = data travelling UP, 1 = DOWN, 2 = LEFT, 3 = RIGHT
def exchange_halos_async(d_u, bufs, ctx, stream_boundary):
    """Stage A (pack + async D2H on stream_boundary) and start of Stage B
    (sync, then post non-blocking Irecv/Isend). Returns the request list."""
    ny, nx = ctx["ny"], ctx["nx"]
    cart = ctx["cart_comm"]
    threads = 256
    blocks = (max(ny, nx) + threads - 1) // threads
    pack_halos_kernel[blocks, threads, stream_boundary](
        d_u, ny, nx, bufs.d_send["up"], bufs.d_send["down"],
        bufs.d_send["left"], bufs.d_send["right"])
    for k in ("up", "down", "left", "right"):
        bufs.d_send[k].copy_to_host(bufs.h_send[k], stream=stream_boundary)
    stream_boundary.synchronize()

    reqs = []
    # receives: what comes from my 'down' neighbour travelled UP (tag 0), etc.
    reqs.append(cart.Irecv([bufs.h_recv["down"], MPI.DOUBLE], source=ctx["down"], tag=0))
    reqs.append(cart.Irecv([bufs.h_recv["up"], MPI.DOUBLE], source=ctx["up"], tag=1))
    reqs.append(cart.Irecv([bufs.h_recv["right"], MPI.DOUBLE], source=ctx["right"], tag=2))
    reqs.append(cart.Irecv([bufs.h_recv["left"], MPI.DOUBLE], source=ctx["left"], tag=3))
    reqs.append(cart.Isend([bufs.h_send["up"], MPI.DOUBLE], dest=ctx["up"], tag=0))
    reqs.append(cart.Isend([bufs.h_send["down"], MPI.DOUBLE], dest=ctx["down"], tag=1))
    reqs.append(cart.Isend([bufs.h_send["left"], MPI.DOUBLE], dest=ctx["left"], tag=2))
    reqs.append(cart.Isend([bufs.h_send["right"], MPI.DOUBLE], dest=ctx["right"], tag=3))
    return reqs


def complete_halos_async(reqs, d_u, bufs, ctx, stream_boundary):
    """Stage C part 1: Waitall, async H2D, unpack into ghost cells."""
    ny, nx = ctx["ny"], ctx["nx"]
    MPI.Request.Waitall(reqs)
    for k in ("up", "down", "left", "right"):
        bufs.d_recv[k].copy_to_device(bufs.h_recv[k], stream=stream_boundary)
    threads = 256
    blocks = (max(ny, nx) + threads - 1) // threads
    unpack_halos_kernel[blocks, threads, stream_boundary](
        d_u, ny, nx, bufs.d_recv["up"], bufs.d_recv["down"],
        bufs.d_recv["left"], bufs.d_recv["right"],
        ctx["up"] != MPI.PROC_NULL, ctx["down"] != MPI.PROC_NULL,
        ctx["left"] != MPI.PROC_NULL, ctx["right"] != MPI.PROC_NULL)


# ==============================================================================
# TASK 4: tiled stencil + orchestrators
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, u_new, kappa, S, dt, inv_dx2, inv_dy2, lam,
                         r_start, r_end, c_start, c_end):
    """16x16 block, 18x18 shared tile. Updates cells with local indices
    r_start..r_end, c_start..c_end (inclusive)."""
    smem = cuda.shared.array(shape=(18, 18), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * 16 + ty
    j = c_start + cuda.blockIdx.x * 16 + tx
    H = u.shape[0]
    W = u.shape[1]

    in_arr = (i < H) and (j < W)
    if in_arr:
        smem[ty + 1, tx + 1] = u[i, j]
        if ty == 0 and i - 1 >= 0:
            smem[0, tx + 1] = u[i - 1, j]
        if ty == 15 and i + 1 < H:
            smem[17, tx + 1] = u[i + 1, j]
        if tx == 0 and j - 1 >= 0:
            smem[ty + 1, 0] = u[i, j - 1]
        if tx == 15 and j + 1 < W:
            smem[ty + 1, 17] = u[i, j + 1]
    cuda.syncthreads()   # reached by every thread (no early return above)

    if i >= r_start and i <= r_end and j >= c_start and j <= c_end:
        c = smem[ty + 1, tx + 1]
        kc = kappa[i, j]
        kn = 0.5 * (kappa[i + 1, j] + kc)
        ks = 0.5 * (kappa[i - 1, j] + kc)
        ke = 0.5 * (kappa[i, j + 1] + kc)
        kw = 0.5 * (kappa[i, j - 1] + kc)
        dyy = (kn * (smem[ty + 2, tx + 1] - c) - ks * (c - smem[ty, tx + 1])) * inv_dx2
        dxx = (ke * (smem[ty + 1, tx + 2] - c) - kw * (c - smem[ty + 1, tx])) * inv_dy2
        u_new[i, j] = c + dt * (dyy + dxx - lam * c * c * c + S[i, j])


def launch_stencil(d_u, d_un, d_k, d_S, p, r0, r1, c0, c1, stream):
    if r1 < r0 or c1 < c0:
        return
    grid = ((c1 - c0 + TILE_DIM) // TILE_DIM, (r1 - r0 + TILE_DIM) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        d_u, d_un, d_k, d_S, p["dt"], p["inv_dx2"], p["inv_dy2"], LAMBDA_DAMP,
        r0, r1, c0, c1)


def step_serialized(d_u, d_un, d_k, d_S, bufs, ctx, p, s_int, s_bnd):
    ny, nx = ctx["ny"], ctx["nx"]
    reqs = exchange_halos_async(d_u, bufs, ctx, s_bnd)
    complete_halos_async(reqs, d_u, bufs, ctx, s_bnd)
    launch_stencil(d_u, d_un, d_k, d_S, p, 1, ny, 1, nx, s_bnd)   # whole interior after exchange
    s_bnd.synchronize()
    s_int.synchronize()


def step_overlapped(d_u, d_un, d_k, d_S, bufs, ctx, p, s_int, s_bnd):
    ny, nx = ctx["ny"], ctx["nx"]
    # Stage A + B: pack, D2H, sync, post Irecv/Isend
    reqs = exchange_halos_async(d_u, bufs, ctx, s_bnd)
    # deep interior on stream_interior, concurrent with MPI traffic
    launch_stencil(d_u, d_un, d_k, d_S, p, 2, ny - 1, 2, nx - 1, s_int)
    # Stage C: wait, H2D, unpack, boundary ring
    complete_halos_async(reqs, d_u, bufs, ctx, s_bnd)
    launch_stencil(d_u, d_un, d_k, d_S, p, 1, 1, 1, nx, s_bnd)            # row 1
    launch_stencil(d_u, d_un, d_k, d_S, p, ny, ny, 1, nx, s_bnd)          # row ny
    launch_stencil(d_u, d_un, d_k, d_S, p, 2, ny - 1, 1, 1, s_bnd)        # column 1
    launch_stencil(d_u, d_un, d_k, d_S, p, 2, ny - 1, nx, nx, s_bnd)      # column nx
    s_bnd.synchronize()
    s_int.synchronize()


# ==============================================================================
# TASK 5
# ==============================================================================
def global_reductions(d_old, d_new, ctx, p, with_residual=True):
    ny, nx = ctx["ny"], ctx["nx"]
    cart = ctx["cart_comm"]
    new = d_new.copy_to_host()[1:ny + 1, 1:nx + 1]
    e_loc = np.array([new.sum() * p["dx"] * p["dy"]])
    if with_residual:
        old = d_old.copy_to_host()[1:ny + 1, 1:nx + 1]
        r_loc = np.array([(((new - old) / p["dt"]) ** 2).sum() * p["dx"] * p["dy"]])
    else:
        r_loc = np.array([0.0])
    e_g = np.zeros(1)
    r_g = np.zeros(1)
    cart.Allreduce(e_loc, e_g, op=MPI.SUM)
    cart.Allreduce(r_loc, r_g, op=MPI.SUM)
    return float(e_g[0]), (math.sqrt(r_g[0]) if with_residual else None)


def run_mode(ctx, mode, steps, report=True):
    comm = ctx["comm"]
    h_u, h_k, h_S, dx, dy = initialize_subdomains(ctx)
    kmax_loc = np.array([h_k[1:-1, 1:-1].max()])
    kmax = np.zeros(1)
    ctx["cart_comm"].Allreduce(kmax_loc, kmax, op=MPI.MAX)
    kmax = float(kmax[0])
    dt_max = dx * dx / (4.0 * kmax)
    p = {"dx": dx, "dy": dy, "dt": CFL_SAFETY * dt_max,
         "inv_dx2": 1.0 / (dx * dx), "inv_dy2": 1.0 / (dy * dy),
         "kmax": kmax, "dt_max": dt_max}

    d_u = cuda.to_device(h_u)
    d_un = cuda.to_device(h_u.copy())
    d_k = cuda.to_device(h_k)
    d_S = cuda.to_device(h_S)
    bufs = HaloBuffers(ctx["ny"], ctx["nx"])
    s_int = cuda.stream()
    s_bnd = cuda.stream()
    step_fn = step_overlapped if mode == "overlapped" else step_serialized

    trace = []
    e0, _ = global_reductions(d_u, d_u, ctx, p, with_residual=False)
    trace.append({"step": 0, "energy": e0, "residual_l2": None})

    cuda.synchronize()
    comm.Barrier()
    ev0 = cuda.event(timing=True)
    ev1 = cuda.event(timing=True)
    t0 = time.perf_counter()
    ev0.record(s_bnd)
    for s in range(1, steps + 1):
        step_fn(d_u, d_un, d_k, d_S, bufs, ctx, p, s_int, s_bnd)
        if s % 100 == 0:
            e, r = global_reductions(d_u, d_un, ctx, p)
            trace.append({"step": s, "energy": e, "residual_l2": r})
        d_u, d_un = d_un, d_u
    ev1.record(s_bnd)
    ev1.synchronize()
    wall_ms = (time.perf_counter() - t0) * 1e3
    ev_ms = cuda.event_elapsed_time(ev0, ev1)
    wall_ms = comm.allreduce(wall_ms, op=MPI.MAX)
    ev_ms = comm.allreduce(ev_ms, op=MPI.MAX)

    if report and ctx["rank"] == 0:
        print(f"[{mode}] kappa_max={kmax:.6f} dt={p['dt']:.6e} dt_max={dt_max:.6e}")
        for t in trace:
            r = "n/a" if t["residual_l2"] is None else f"{t['residual_l2']:.6e}"
            print(f"[{mode}] step {t['step']:4d}  E={t['energy']:.12e}  L2res={r}")
        print(f"[{mode}] wall={wall_ms:.3f} ms   cuda_event={ev_ms:.3f} ms")

    return {"final": d_u.copy_to_host(), "trace": trace, "wall_ms": wall_ms,
            "event_ms": ev_ms, "p": p}


def ptx_audit():
    counts = {"st.shared.f64": 0, "ld.shared.f64": 0, "bar.sync": 0}
    asm = tiled_stencil_kernel.inspect_asm()
    for ptx in asm.values():
        for key in counts:
            counts[key] += len(re.findall(re.escape(key), ptx))
    passed = all(v > 0 for v in counts.values())
    return counts, passed


def run_solver(github_user):
    ctx = init_distributed_context(github_user)
    rank = ctx["rank"]
    if rank == 0:
        print(f"GITHUB_USERNAME={github_user}")
        print(f"GPU_UUID={ctx['gpu_uuid']}")
        print(f"INSTANCE_SEED={ctx['seed']}")
        print(f"MPI ranks={ctx['size']} (2x2 Cartesian, non-periodic)")
    print(f"rank {rank}: coords={ctx['coords']} up={ctx['up']} down={ctx['down']} "
          f"left={ctx['left']} right={ctx['right']}", flush=True)

    # warm-up (JIT compile, context creation) - not timed
    run_mode(ctx, "serialized", 2, report=False)
    run_mode(ctx, "overlapped", 2, report=False)

    ser = run_mode(ctx, "serialized", STEPS)
    ovl = run_mode(ctx, "overlapped", STEPS)

    ny, nx = ctx["ny"], ctx["nx"]
    diff_loc = np.array([np.max(np.abs(ser["final"][1:ny + 1, 1:nx + 1]
                                       - ovl["final"][1:ny + 1, 1:nx + 1]))])
    diff = np.zeros(1)
    ctx["cart_comm"].Allreduce(diff_loc, diff, op=MPI.MAX)
    parity = float(diff[0])

    if rank == 0:
        counts, passed = ptx_audit()
        print(f"PTX counts: {counts}  ptx_audit_passed={passed}")
        print(f"parity_max_diff={parity:.3e}")
        print(f"serialized_ms={ser['wall_ms']:.3f}  overlapped_ms={ovl['wall_ms']:.3f}")
        manifest = {
            "github_username": github_user,
            "gpu_uuid": ctx["gpu_uuid"],
            "instance_seed": ctx["seed"],
            "mpi_ranks": ctx["size"],
            "process_grid": [2, 2],
            "global_grid": [GLOBAL_NY, GLOBAL_NX],
            "steps": STEPS,
            "dt": ser["p"]["dt"],
            "dt_max": ser["p"]["dt_max"],
            "kappa_max": ser["p"]["kmax"],
            "serialized_ms": ser["wall_ms"],
            "overlapped_ms": ovl["wall_ms"],
            "cuda_event_serialized_ms": ser["event_ms"],
            "cuda_event_overlapped_ms": ovl["event_ms"],
            "energy_trace_serialized": ser["trace"],
            "energy_trace_overlapped": ovl["trace"],
            "ptx_counts": counts,
            "ptx_audit_passed": bool(passed),
            "parity_max_diff": parity,
            "parity_ok": parity < 1e-12,
        }
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print("Wrote submission_manifest.json")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise SystemExit("usage: distributed_cuda_solver.py <github_username>")
    run_solver(sys.argv[1])


Overwriting distributed_cuda_solver.py


In [20]:
import subprocess
cmd = ["mpirun", "--allow-run-as-root", "--oversubscribe", "-np", "4",
       "python3", "distributed_cuda_solver.py", GITHUB_USERNAME]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")


GITHUB_USERNAME=AliOrynbay
GPU_UUID=GPU-5d365dc5-35ff-82b3-ce91-bf085d8f2f7b
INSTANCE_SEED=11006944976712755476
MPI ranks=4 (2x2 Cartesian, non-periodic)
rank 0: coords=(0, 0) up=-2 down=2 left=-2 right=1
rank 1: coords=(0, 1) up=-2 down=3 left=0 right=-2
rank 2: coords=(1, 0) up=0 down=-2 left=-2 right=3
rank 3: coords=(1, 1) up=1 down=-2 left=2 right=-2
[serialized] kappa_max=0.239997 dt=3.166572e-06 dt_max=3.958215e-06
[serialized] step    0  E=5.559278977593e-02  L2res=n/a
[serialized] step  100  E=5.608952789289e-02  L2res=1.283501e+01
[serialized] step  200  E=5.658629043340e-02  L2res=1.243734e+01
[serialized] step  300  E=5.708306946875e-02  L2res=1.206950e+01
[serialized] step  400  E=5.757986062182e-02  L2res=1.172762e+01
[serialized] wall=1662.670 ms   cuda_event=1662.673 ms
[overlapped] kappa_max=0.239997 dt=3.166572e-06 dt_max=3.958215e-06
[overlapped] step    0  E=5.559278977593e-02  L2res=n/a
[overlapped] step  100  E=5.608952789289e-02  L2res=1.283501e+01
[overlapped] s

- **Question 1 (Memory Coalescing & Strided Column Packing):**
In row-major storage, `u[i,1]` and `u[i+1,1]` are 258 × 8 B = 2064 B apart. A warp reading a column therefore touches 32 different 128-byte lines and uses only 8 B of each, so every load is a separate transaction and nothing coalesces. A DMA engine moves one contiguous span (or a 2D pitched copy), so a column cannot be sent in one plain copy. A single contiguous bounding-box copy from `u[1,1]` to `u[256,1]` would span (255·258 + 1)·8 = 526,328 B, of which only 256·8 = 2,048 B are real data. The extra traffic is 524,280 B (about 257×) per column, and about 1.05 MB per step for both columns. Rows are contiguous, so they cost exactly 2,048 B. `pack_halos_kernel` reads the strided column on the GPU and writes it coalesced into a 2 KB buffer, so only 2,048 B per strip cross PCIe.

- **Question 2 (Shared-Memory Tile Indexing & Bank Conflict Analysis):**
Thread (tx=0, ty=0) writes `smem[1,1]` (center), `smem[0,1]` (ty==0 apron) and `smem[1,0]` (tx==0 apron). It reads `smem[1,1]`, `[2,1]`, `[0,1]`, `[1,2]` and `[1,0]`. Thread (tx=15, ty=15) writes `smem[16,16]`, `smem[17,16]` (ty==15) and `smem[16,17]` (tx==15). It reads `smem[16,16]`, `[17,16]`, `[15,16]`, `[16,17]` and `[16,15]`. On CC 7.5 a float64 occupies 2 consecutive 4-byte banks. A warp is two rows of 16 threads with consecutive `tx`, and 64-bit accesses are serviced per half-warp: 16 threads × 8 B = 128 B covers all 32 banks exactly once, so row accesses are conflict-free (the 256 B per warp costs two wavefronts, which is inherent, not a conflict). The north/south reads (`smem[ty±1, tx+1]`) also vary along `tx`, so they are conflict-free as well. A 2-way conflict would appear only for column-wise access (fixed `tx`, varying `ty`): the row pitch is 18 × 8 B = 36 words, which is 4 banks mod 32, so rows `ty` and `ty+8` map to the same banks. This affects only the few `tx==0` / `tx==15` apron stores.

- **Question 3 (PTX Assembly Audit & Warp Synchronization):**
From my run, `tiled_stencil_kernel.inspect_asm()` contains `st.shared.f64 = 5`, `ld.shared.f64 = 5`, `bar.sync = 1` (`ptx_audit_passed = True`). These are static instruction counts. The 5 stores are the center cell plus the four apron stores (north, south, west, east). The 5 loads are the center value plus its four neighbours. The single `bar.sync` is the `cuda.syncthreads()` between apron loading and stencil arithmetic. A 16×16 block has 8 warps (each covers two `ty` rows). Without the barrier there is a read-after-write hazard across warps: a thread reads `smem[ty±1, ...]`, which was written by a thread in another warp (e.g. row ty=2 reads the row written by ty=1 in warp 0, and the apron rows/columns are written only by the ty==0/ty==15/tx==0/tx==15 threads). Warps are scheduled independently, so the reader may see stale or uninitialised shared memory, giving nondeterministic results.

- **Question 4 (Multi-Process Context Sharing on a Single Physical GPU):**
Each of the four MPI processes owns its own CUDA context. Without MPS the driver time-slices contexts: kernels from different processes do not run concurrently on the SMs, and copy-engine work is also arbitrated between contexts. Measured in my final run: `serialized_ms = 1662.670` and `overlapped_ms = 2076.603` for 400 steps (4.16 vs 5.19 ms per step), i.e. the overlapped version was 413.9 ms slower (0.80×, about 25% slower). The CUDA-event times (1662.673 / 2076.498 ms) match the wall times, so the time is spent on the GPU/MPI timeline and not in host-side idle. Overlap does not help here for several reasons: (a) intra-node shared-memory MPI latency is tiny, so there is almost no communication to hide; (b) each 256×256 tile is a short kernel, so per-launch overhead dominates and the overlapped step issues 5 stencil launches (1 interior + 4 ring) instead of 1, plus extra stream synchronisations; (c) the four contexts compete for the same SMs and DMA engines, so the idle time overlap would exploit is taken by other ranks; (d) the ring launches use 16×16 blocks on 1-cell-wide strips, so most threads are idle. In an earlier run of the same code the overlapped version was about 10% faster (2252.950 ms serialized vs 2029.206 ms overlapped), so on a shared Colab T4 the sign of the difference varies from run to run and the single-GPU benefit is small and unreliable. With four dedicated GPUs over InfiniBand, each rank owns its SMs and copy engines, and network latency and bandwidth are large compared with a small kernel, so overlapping the interior kernel with Isend/Irecv would hide much more and give a clear gain.

- **Question 5 (Point-to-Point Message Deadlock & MPI.PROC_NULL Semantics):**
MPI sends below an eager threshold (a few KB for shared memory, tunable) return as soon as the data is buffered. Above it, MPI uses the rendezvous protocol, where `Send` blocks until the matching `Recv` is posted. On large grids the halo messages exceed the threshold. If neighbouring ranks both call blocking `Send` toward each other before either posts `Recv` (e.g. `Send(up)`, `Send(down)`, then `Recv`), each waits for the other and the program deadlocks. On a periodic grid a fixed directional order creates a circular wait around the ring. Non-blocking `Isend/Irecv` plus `Waitall` removes this, because all receives are posted before waiting. `MPI.PROC_NULL` turns a send or receive to a missing neighbour into a no-op that completes immediately and leaves the receive buffer unchanged. Rank (0,0) has `up = -2` and `left = -2` in my run (PROC_NULL), so the same code runs without `if` branches, and the Dirichlet zero ghost cells are never overwritten (my unpack also skips PROC_NULL sides).

- **Question 6 (Explicit CFL Stability & Energy Conservation Proof):**
Δx = 1/513, so Δx² = 3.800e-6. My run printed κmax = 0.239997, so Δt_max = Δx²/(4κmax) = 3.958215e-6. The simulation uses Δt = 3.166572e-6 = 0.8·Δt_max, which satisfies Δt < Δt_max. The Allreduce energy trace is E(0)=5.559279e-2, E(100)=5.608953e-2, E(200)=5.658629e-2, E(300)=5.708307e-2, E(400)=5.757986e-2. It is positive, smooth and monotone, with no oscillation or growth in magnitude, as an unstable scheme would show. Energy is not conserved (there is a source, cubic damping and Dirichlet losses). It rises almost linearly by about 4.967e-4 per 100 steps, i.e. dE/dt ≈ 4.967e-4 / (100·3.1666e-6) ≈ 1.569, which matches the integrated source strength ∫S dx dy = 100·2π·0.05² ≈ 1.571. Diffusion lengths over t = 1.27e-3 are small (about 0.03), so boundary losses and cubic damping are negligible in this short run. The L2 residual decreases monotonically (12.835 → 12.437 → 12.070 → 11.728), consistent with a stable evolution.